# Experiment 3 Caching Memory Sweeps

This notebook sweeps **SRAM size (GlobalBuffer, MB)** and **MainMemory size (GB)** and generates policy-wise heatmaps.

For each policy, it outputs:
- Energy heatmap
- Latency heatmap

Axes:
- **x-axis**: SRAM size (MB)
- **y-axis**: MainMemory size (GB)

Heatmaps are averaged across configured datasets/reuse ratios.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scripts.cache_sim import (
    simulate_lfu_cost_aware,
    simulate_no_cache,
    simulate_two_tier_lfu,
    simulate_two_tier_opt,
    synth_trace,
)

WORKSPACE = Path.cwd()
OUT_DIR = WORKSPACE / "new_exp3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sweep axes
SRAM_MB_VALUES = [32, 64, 128, 256, 512]
MAIN_MEMORY_GB_VALUES = [1, 2, 4, 8, 16]

# Trace settings (same structure as Exp3)
N_DOCS_TRACE = 2_000_000
N_QUERIES = 100_000
K_SIM = 10
DECAY_FACTOR = 0.99
SEED = 42

BEIR_DATASETS = [
    ("nq", 1.25),
    ("hotpotqa", 1.42),
    ("scidocs", 1.73),
    ("quora", 1.91),
    ("fever", 2.41),
    ("fiqa", 4.47),
]

POLICIES = ["no_cache", "lfu_l1", "lfu_l2", "lfu_l1_l2", "opt_two_tier"]
POLICY_LABELS = {
    "no_cache": "No Cache",
    "lfu_l1": "LFU L1 (SRAM)",
    "lfu_l2": "DRAM LFU",
    "lfu_l1_l2": "LFU L1/L2",
    "opt_two_tier": "OPT L1/L2",
}

# EdgeRAG-aligned retrieval cost model
ENC_EMBED_DIM = 768
BITS_PER_VAL = 8
EMB_BITS = ENC_EMBED_DIM * BITS_PER_VAL
EMB_BYTES = EMB_BITS // 8

E_L2_PJ = 1.88 * EMB_BITS
E_DRAM_PJ = 7.03 * EMB_BITS
E_DISK_PJ = 10e-9 * 1e12
T_L2_NS = EMB_BITS / (2048.0 * 1e9 * 8) * 1e9
T_DRAM_NS = EMB_BITS / (614.0 * 1e9 * 8) * 1e9
T_DISK_NS = 100e-6 * 1e9


def embedding_capacity_bytes(capacity_bytes: int, n_docs: int) -> int:
    return min(int(capacity_bytes) // EMB_BYTES, n_docs)


def sram_capacity(sram_mb: float, n_docs: int) -> int:
    return embedding_capacity_bytes(sram_mb * 1024**2, n_docs)


def dram_capacity(dram_gb: float, n_docs: int) -> int:
    return embedding_capacity_bytes(dram_gb * 1024**3, n_docs)


def metrics_from_tiers(h2: int, h_dram: int, h_disk: int) -> dict[str, float]:
    e_pj = h2 * E_L2_PJ + h_dram * (E_L2_PJ + E_DRAM_PJ) + h_disk * (E_L2_PJ + E_DRAM_PJ + E_DISK_PJ)
    t_ns = h2 * T_L2_NS + h_dram * (T_L2_NS + T_DRAM_NS) + h_disk * (T_L2_NS + T_DRAM_NS + T_DISK_NS)
    return {
        "energy_nJ_per_query": e_pj / N_QUERIES / 1e3,
        "latency_us_per_query": t_ns / N_QUERIES / 1e3,
    }


print("Configured. Run next cells when ready.")

In [ ]:
def run_memory_sweep() -> pd.DataFrame:
    rows = []

    for dataset, reuse in BEIR_DATASETS:
        trace = synth_trace(
            n_queries=N_QUERIES,
            n_docs=N_DOCS_TRACE,
            reuse_ratio=reuse,
            k_per_query=K_SIM,
            seed=SEED,
        )

        for dram_gb in MAIN_MEMORY_GB_VALUES:
            l2_cap = dram_capacity(dram_gb, N_DOCS_TRACE)

            for sram_mb in SRAM_MB_VALUES:
                l1_cap = sram_capacity(sram_mb, N_DOCS_TRACE)

                no_cache = simulate_no_cache(trace)
                lfu_l1 = simulate_lfu_cost_aware(trace, l1_cap, gen_latency={}, decay_factor=DECAY_FACTOR)
                lfu_l2 = simulate_lfu_cost_aware(trace, l2_cap, gen_latency={}, decay_factor=DECAY_FACTOR)
                lfu_l1_l2 = simulate_two_tier_lfu(
                    trace,
                    l1_capacity=l1_cap,
                    l2_capacity=l2_cap,
                    gen_latency={},
                    decay_factor=DECAY_FACTOR,
                    check_invariants=False,
                )
                opt_l1_l2 = simulate_two_tier_opt(
                    trace,
                    l1_capacity=l1_cap,
                    l2_capacity=l2_cap,
                    check_invariants=False,
                )

                policy_counts = {
                    "no_cache": {"h2": 0, "h_dram": 0, "h_disk": no_cache.misses},
                    "lfu_l1": {"h2": lfu_l1.hits, "h_dram": 0, "h_disk": lfu_l1.misses},
                    "lfu_l2": {"h2": 0, "h_dram": lfu_l2.hits, "h_disk": lfu_l2.misses},
                    "lfu_l1_l2": {"h2": lfu_l1_l2.l1_hits, "h_dram": lfu_l1_l2.l2_hits, "h_disk": lfu_l1_l2.misses},
                    "opt_two_tier": {"h2": opt_l1_l2.l1_hits, "h_dram": opt_l1_l2.l2_hits, "h_disk": opt_l1_l2.misses},
                }

                for policy, c in policy_counts.items():
                    total = c["h2"] + c["h_dram"] + c["h_disk"]
                    metrics = metrics_from_tiers(c["h2"], c["h_dram"], c["h_disk"])
                    rows.append(
                        {
                            "dataset": dataset,
                            "reuse_target": reuse,
                            "main_memory_gb": dram_gb,
                            "sram_mb": sram_mb,
                            "policy": policy,
                            "policy_label": POLICY_LABELS[policy],
                            "h2": c["h2"],
                            "h_dram": c["h_dram"],
                            "h_disk": c["h_disk"],
                            "hit_rate": (c["h2"] + c["h_dram"]) / total if total else 0.0,
                            **metrics,
                        }
                    )

    return pd.DataFrame(rows)


# Run this cell to produce sweep data.
sweep_df = run_memory_sweep()
sweep_csv = OUT_DIR / "exp3_memory_sweep_raw.csv"
sweep_df.to_csv(sweep_csv, index=False)
print(f"Wrote {sweep_csv}")
sweep_df.head()

In [ ]:
def plot_policy_heatmaps(df: pd.DataFrame) -> None:
    agg = (
        df.groupby(["policy", "policy_label", "main_memory_gb", "sram_mb"], as_index=False)
        .agg(
            energy_nJ_per_query=("energy_nJ_per_query", "mean"),
            latency_us_per_query=("latency_us_per_query", "mean"),
        )
    )

    for policy in POLICIES:
        sub = agg[agg["policy"] == policy].copy()
        if sub.empty:
            continue

        label = sub["policy_label"].iloc[0]
        energy_grid = (
            sub.pivot(index="main_memory_gb", columns="sram_mb", values="energy_nJ_per_query")
            .reindex(index=MAIN_MEMORY_GB_VALUES, columns=SRAM_MB_VALUES)
        )
        latency_grid = (
            sub.pivot(index="main_memory_gb", columns="sram_mb", values="latency_us_per_query")
            .reindex(index=MAIN_MEMORY_GB_VALUES, columns=SRAM_MB_VALUES)
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
        fig.suptitle(f"Experiment 3 memory sweeps: {label}")

        im0 = axes[0].imshow(energy_grid.to_numpy(), aspect="auto", origin="lower", cmap="viridis")
        axes[0].set_title("Energy [nJ/query]")
        axes[0].set_xlabel("SRAM size [MB]")
        axes[0].set_ylabel("MainMemory size [GB]")
        axes[0].set_xticks(np.arange(len(SRAM_MB_VALUES)))
        axes[0].set_xticklabels(SRAM_MB_VALUES)
        axes[0].set_yticks(np.arange(len(MAIN_MEMORY_GB_VALUES)))
        axes[0].set_yticklabels(MAIN_MEMORY_GB_VALUES)
        fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

        im1 = axes[1].imshow(latency_grid.to_numpy(), aspect="auto", origin="lower", cmap="magma")
        axes[1].set_title("Latency [us/query]")
        axes[1].set_xlabel("SRAM size [MB]")
        axes[1].set_ylabel("MainMemory size [GB]")
        axes[1].set_xticks(np.arange(len(SRAM_MB_VALUES)))
        axes[1].set_xticklabels(SRAM_MB_VALUES)
        axes[1].set_yticks(np.arange(len(MAIN_MEMORY_GB_VALUES)))
        axes[1].set_yticklabels(MAIN_MEMORY_GB_VALUES)
        fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

        out = OUT_DIR / f"exp3_memory_sweep_{policy}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Wrote {out}")


# Run this cell after sweep_df exists.
plot_policy_heatmaps(sweep_df)

## Notes

- This notebook is **set up only**; it has not been executed.
- Output files are written to `workspace/new_exp3` by default.
- If runtime is high, reduce `N_QUERIES` or the sweep grid sizes before running.